# SOCO: DINOv2 ViT-B/14 LoRA v2.1 (self-contained, Colab)

Trains a LoRA adapter (qkv, proj, fc1, fc2 in the last 4 blocks, plus trainable LayerNorms) on the AML competition `train.csv`, validates on an image-disjoint split, and writes a submission CSV.

**Before you run:** Runtime -> Change runtime type -> GPU (T4 is enough). You need the competition files (`train.csv`, `test.csv`, `sample_submission.csv`, and the image folders) in one folder. Set `DATA_ROOT` in the first code cell.

Recipe (from the original Kaggle run): train at 644 px, 4 epochs, batch 4, lr 8e-5, tau 0.03, LoRA rank 8 / alpha 16 / dropout 0.05, evaluate and predict at 840 px with a soft-window (5) soft-argmax at temperature 0.08. The original run scored 59.76% PCK@0.1 on its 502-query validation split. Results will not be bit-identical on Colab (different library versions and GPU nondeterminism).

In [ ]:
import os, sys, json, math, random, time, gc, copy, warnings
from pathlib import Path
from collections import OrderedDict
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", message="xFormers")
assert torch.cuda.is_available(), "Enable a GPU: Runtime -> Change runtime type -> T4 GPU"
DEVICE = "cuda"

# ===================== EDIT THIS =====================
# Folder that contains train.csv, test.csv, sample_submission.csv and the image folders.
DATA_ROOT = Path("/content/drive/MyDrive/aml-challenge-2026-27")
MOUNT_DRIVE = True          # set False if the data is already under /content
# =====================================================

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OUT_DIR = Path("/content/outputs"); OUT_DIR.mkdir(parents=True, exist_ok=True)
for f in ["train.csv", "test.csv", "sample_submission.csv"]:
    assert (DATA_ROOT / f).is_file(), f"Missing {DATA_ROOT / f}"

HP = dict(
    train_size=644, eval_size=840,
    epochs=4, batch_size=4, lr=8e-5, tau=0.03, weight_decay=0.05,
    lora_rank=8, lora_alpha=16.0, lora_dropout=0.05, n_blocks=4,
    soft_window=5, soft_temperature=0.08,
    seed=0, val_fraction=0.20,
)
print("GPU:", torch.cuda.get_device_name(0))
print(HP)

In [ ]:
class QueryDataset(Dataset):
    # one source keypoint per row; train rows also carry the accepted target points and a scale
    def __init__(self, df, root):
        self.df = df.reset_index(drop=True).copy()
        self.root = Path(root)
        self.labeled = "target_points" in self.df.columns

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        sp, tp = self.root / r["source_image"], self.root / r["target_image"]
        with Image.open(sp) as im: src = im.convert("RGB")
        with Image.open(tp) as im: trg = im.convert("RGB")
        s = {
            "id": str(r["id"]), "src_path": str(sp), "trg_path": str(tp),
            "src_img": src, "trg_img": trg,
            "src_kps": np.asarray([[r["source_x"], r["source_y"]]], dtype=np.float32),
        }
        if self.labeled:
            t = np.asarray(json.loads(r["target_points"]), dtype=np.float32).reshape(-1, 2)
            assert len(t) > 0 and np.isfinite(t).all()
            s["accepted_targets"] = t
            s["scale"] = float(r["scale"])
        return s

train_all = pd.read_csv(DATA_ROOT / "train.csv").reset_index(drop=True)
src_keys = train_all["source_image"].map(lambda p: str((DATA_ROOT / str(p)).resolve()))
trg_keys = train_all["target_image"].map(lambda p: str((DATA_ROOT / str(p)).resolve()))

# Union-Find: images connected through any pair (or any shared pair_id) form one group,
# so no image appears in both train and validation.
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    a, b = find(a), find(b)
    if a != b:
        parent[b] = a
pair_anchor = {}
for pid, s_, t_ in zip(train_all["pair_id"], src_keys, trg_keys):
    union(s_, t_)
    if pid in pair_anchor: union(s_, pair_anchor[pid])
    else: pair_anchor[pid] = s_
groups = np.asarray([find(s_) for s_ in src_keys])

splitter = GroupShuffleSplit(n_splits=100, test_size=HP["val_fraction"], random_state=42)
tr_idx, va_idx = min(splitter.split(train_all, groups=groups),
                     key=lambda ix: abs(len(ix[1]) / len(train_all) - HP["val_fraction"]))
train_df = train_all.iloc[tr_idx].copy().reset_index(drop=True)
val_df = train_all.iloc[va_idx].sample(frac=1, random_state=42).reset_index(drop=True)

train_ds, val_ds = QueryDataset(train_df, DATA_ROOT), QueryDataset(val_df, DATA_ROOT)
test_ds = QueryDataset(pd.read_csv(DATA_ROOT / "test.csv"), DATA_ROOT)

tr_imgs = set(train_df["source_image"]) | set(train_df["target_image"])
va_imgs = set(val_df["source_image"]) | set(val_df["target_image"])
assert tr_imgs.isdisjoint(va_imgs), "Image overlap between train and val!"
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}   (original Kaggle run: 1993 / 502 / 577)")

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def preprocess(img, size):
    arr = np.asarray(img.resize((size, size), Image.BICUBIC), dtype=np.float32) / 255.0
    t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    return (t - IMAGENET_MEAN) / IMAGENET_STD

# ---------------- LoRA ----------------
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad_(False)
        self.scale = alpha / rank
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        dev, dt = base.weight.device, base.weight.dtype
        self.lora_A = nn.Parameter(torch.empty((rank, base.in_features), device=dev, dtype=dt))
        self.lora_B = nn.Parameter(torch.zeros((base.out_features, rank), device=dev, dtype=dt))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.lora_A), self.lora_B) * self.scale

def inject_lora(blocks, n_blocks, rank, alpha, dropout):
    wrapped = 0
    for blk in list(blocks)[-n_blocks:]:
        for parent_mod, names in ((blk.attn, ("qkv", "proj")), (blk.mlp, ("fc1", "fc2"))):
            for n in names:
                layer = getattr(parent_mod, n, None)
                if isinstance(layer, nn.Linear):
                    setattr(parent_mod, n, LoRALinear(layer, rank, alpha, dropout)); wrapped += 1
    return wrapped

def build_model(hp, lora=True):
    model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14", trust_repo=True)   # downloads pretrained weights
    for p in model.parameters(): p.requires_grad_(False)
    if lora:
        n = inject_lora(model.blocks, hp["n_blocks"], hp["lora_rank"], hp["lora_alpha"], hp["lora_dropout"])
        assert n == 4 * hp["n_blocks"], f"expected {4 * hp['n_blocks']} wrapped layers, got {n}"
        for m in model.modules():
            if isinstance(m, LoRALinear):
                m.lora_A.requires_grad_(True); m.lora_B.requires_grad_(True)
        for blk in list(model.blocks)[-hp["n_blocks"]:]:
            for m in blk.modules():
                if isinstance(m, nn.LayerNorm):
                    for p in m.parameters(): p.requires_grad_(True)
        for p in model.norm.parameters(): p.requires_grad_(True)
    return model.to(DEVICE)

def trainable_state(model):
    keys = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keys}

def features(model, x):
    # last block, normalized patch features [B, C, S/14, S/14], L2-normalized over channels
    idx = len(model.blocks) - 1
    f = model.get_intermediate_layers(x, n=[idx], reshape=True, norm=True)[0]
    return F.normalize(f, dim=1)

# ---------------- matching ----------------
def sample_features(feat, kps, size):
    grid = (kps / size) * 2 - 1
    out = F.grid_sample(feat[None], grid[None, None], mode="bilinear", align_corners=False)
    return F.normalize(out[0, :, 0].T, dim=-1)

def cell_centers(n, size, device):
    return (torch.arange(n, device=device, dtype=torch.float32) + 0.5) * (size / n)

def locate(sim, size, window, tau):
    # windowed soft-argmax around the peak of each [H, W] similarity map; returns (x, y) in resized pixels
    K, H, W = sim.shape
    cx, cy = cell_centers(W, size, sim.device), cell_centers(H, size, sim.device)
    flat = sim.reshape(K, -1).argmax(-1)
    r, c = flat // W, flat % W
    half = window // 2
    offs = torch.arange(-half, half + 1, device=sim.device)
    rr = (r[:, None] + offs[None]).clamp(0, H - 1)
    cc = (c[:, None] + offs[None]).clamp(0, W - 1)
    patch = sim[torch.arange(K, device=sim.device)[:, None, None], rr[:, :, None], cc[:, None, :]]
    wts = torch.softmax(patch.reshape(K, -1) / tau, -1).reshape(K, len(offs), len(offs))
    x = (wts.sum(1) * cx[cc]).sum(-1)
    y = (wts.sum(2) * cy[rr]).sum(-1)
    return torch.stack([x, y], -1)

class FeatCache:
    def __init__(self, max_items=128):
        self.d, self.max = OrderedDict(), max_items
    def get(self, k):
        if k in self.d: self.d.move_to_end(k); return self.d[k]
        return None
    def put(self, k, v):
        self.d[k] = v
        if len(self.d) > self.max: self.d.popitem(last=False)

@torch.no_grad()
def predict_xy(model, s, S, hp, cache):
    def feat(img, path):
        f = cache.get((path, S))
        if f is None:
            f = features(model, preprocess(img, S).to(DEVICE))[0]; cache.put((path, S), f)
        return f
    fs, ft = feat(s["src_img"], s["src_path"]), feat(s["trg_img"], s["trg_path"])
    k = torch.as_tensor(s["src_kps"], dtype=torch.float32, device=fs.device).clone()
    k[:, 0] *= S / s["src_img"].size[0]; k[:, 1] *= S / s["src_img"].size[1]
    sim = torch.einsum("kc,chw->khw", sample_features(fs, k, S), ft)
    xy = locate(sim, S, hp["soft_window"], hp["soft_temperature"])
    xy[:, 0] *= s["trg_img"].size[0] / S; xy[:, 1] *= s["trg_img"].size[1] / S
    return xy[0].cpu().numpy()

@torch.no_grad()
def evaluate(model, ds, S, hp, desc="eval"):
    # competition metric: distance to the NEAREST accepted target, divided by the row's scale
    model.eval(); cache = FeatCache(); d = []
    for i in tqdm(range(len(ds)), desc=desc, leave=False):
        s = ds[i]
        xy = predict_xy(model, s, S, hp, cache)
        d.append(np.linalg.norm(s["accepted_targets"].astype(np.float64) - xy[None], axis=1).min() / s["scale"])
    d = np.asarray(d)
    return {f"pck@{a}": 100.0 * float((d < a).mean()) for a in (0.05, 0.1, 0.15)}

# ---------------- training loss ----------------
def positive_patch_mask(targets, target_wh, grid_hw, device, radius=1.0):
    w_img, h_img = map(float, target_wh); h, w = map(int, grid_hw)
    pts = torch.as_tensor(targets, dtype=torch.float32, device=device)
    uv = pts / pts.new_tensor([w_img, h_img]) * pts.new_tensor([w, h])
    yy, xx = torch.meshgrid(torch.arange(h, device=device, dtype=torch.float32) + 0.5,
                            torch.arange(w, device=device, dtype=torch.float32) + 0.5, indexing="ij")
    d2 = (xx[None] - uv[:, 0, None, None]).square() + (yy[None] - uv[:, 1, None, None]).square()
    mask = (d2 <= radius ** 2).any(dim=0)
    mask.view(-1)[d2.flatten(1).argmin(dim=1)] = True      # nearest patch of every accepted target
    return mask.flatten()

def matching_loss(fs, ft, src_k, accepted_targets, target_wh, S, tau):
    q = sample_features(fs, src_k.to(fs.device), S)
    logits = (q @ ft.flatten(1)).reshape(-1) / tau
    mask = positive_patch_mask(accepted_targets, target_wh, ft.shape[-2:], ft.device)
    return -torch.logsumexp(F.log_softmax(logits.float(), dim=0)[mask], dim=0)   # -log P(any positive patch)

def flip_sample(s):
    # mirror BOTH images, the source keypoint AND the accepted targets
    s = dict(s)
    ws, wt = float(s["src_img"].size[0]), float(s["trg_img"].size[0])
    s["src_img"] = s["src_img"].transpose(Image.FLIP_LEFT_RIGHT)
    s["trg_img"] = s["trg_img"].transpose(Image.FLIP_LEFT_RIGHT)
    k = s["src_kps"].copy(); k[:, 0] = ws - k[:, 0]; s["src_kps"] = k
    t = s["accepted_targets"].copy(); t[:, 0] = wt - t[:, 0]; s["accepted_targets"] = t
    return s

def to_resized(kps, wh, S):
    k = torch.as_tensor(kps, dtype=torch.float32).clone()
    k[:, 0] *= S / wh[0]; k[:, 1] *= S / wh[1]
    return k

def keep(batch): return batch
print("core functions defined")

In [ ]:
def train(hp, train_ds, val_ds):
    torch.manual_seed(hp["seed"]); random.seed(hp["seed"]); np.random.seed(hp["seed"])
    model = build_model(hp)
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"trainable params: {sum(p.numel() for p in params):,}")
    opt = torch.optim.AdamW(params, lr=hp["lr"], weight_decay=hp["weight_decay"])
    loader = DataLoader(train_ds, batch_size=hp["batch_size"], shuffle=True, num_workers=2,
                        collate_fn=keep, drop_last=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=hp["lr"], total_steps=hp["epochs"] * len(loader), pct_start=0.10)
    scaler = torch.amp.GradScaler("cuda")
    S = hp["train_size"]
    best, best_sd, history = -1.0, None, []
    for ep in range(hp["epochs"]):
        model.train()
        pbar = tqdm(loader, desc=f"epoch {ep}")
        for batch in pbar:
            batch = [flip_sample(s) if random.random() < 0.5 else s for s in batch]
            xs = torch.cat([preprocess(s["src_img"], S) for s in batch]).to(DEVICE)
            xt = torch.cat([preprocess(s["trg_img"], S) for s in batch]).to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                FS, FT = features(model, xs), features(model, xt)
            FS, FT = FS.float(), FT.float()
            loss = sum(matching_loss(FS[b], FT[b], to_resized(s["src_kps"], s["src_img"].size, S),
                                     s["accepted_targets"], s["trg_img"].size, S, hp["tau"])
                       for b, s in enumerate(batch)) / len(batch)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            pbar.set_postfix(loss=f"{loss.item():.3f}")
        m = evaluate(model, val_ds, S, hp, desc=f"val ep{ep}")
        history.append({"epoch": ep, **m})
        print(f"epoch {ep}: val@{S}px  PCK@0.1 = {m['pck@0.1']:.2f}  PCK@0.05 = {m['pck@0.05']:.2f}")
        if m["pck@0.1"] > best:
            best, best_sd = m["pck@0.1"], trainable_state(model)
            torch.save(best_sd, OUT_DIR / "lora_best.pth")
    json.dump(history, open(OUT_DIR / "history.json", "w"), indent=2)
    return model, best_sd, best

# To skip training and reuse a saved adapter, set CKPT to its path and re-run from here.
CKPT = None   # e.g. "/content/drive/MyDrive/lora_best.pth"
if CKPT is None:
    model, best_sd, best_val = train(HP, train_ds, val_ds)
    print(f"best val PCK@0.1 at {HP['train_size']}px: {best_val:.2f}")
else:
    model = build_model(HP)
    best_sd = torch.load(CKPT, map_location="cpu", weights_only=True)

In [ ]:
# Load the best-epoch adapter and evaluate at the higher test resolution
missing, unexpected = model.load_state_dict(best_sd, strict=False)
assert not unexpected, unexpected
m840 = evaluate(model, val_ds, HP["eval_size"], HP, desc="val 840")
print(f"val @ {HP['eval_size']}px: PCK@0.1 = {m840['pck@0.1']:.2f} | PCK@0.05 = {m840['pck@0.05']:.2f} | PCK@0.15 = {m840['pck@0.15']:.2f}")

# Predict the test set
model.eval(); cache = FeatCache(); ids, preds = [], []
for i in tqdm(range(len(test_ds)), desc="test"):
    s = test_ds[i]
    xy = predict_xy(model, s, HP["eval_size"], HP, cache)
    ids.append(s["id"]); preds.append(f"[{round(float(xy[0]), 2)},{round(float(xy[1]), 2)}]")

ref = pd.read_csv(DATA_ROOT / "sample_submission.csv")
sub = pd.DataFrame({"id": ids, "prediction": preds})
assert sub["id"].is_unique and set(sub["id"]) == set(ref["id"].astype(str)), "ids do not match sample_submission"
sub = sub.set_index("id").loc[ref["id"].astype(str)].reset_index()
sub["id"] = ref["id"].values
out_csv = OUT_DIR / "submission_lora_v21_840.csv"
sub.to_csv(out_csv, index=False)
print(len(sub), "rows ->", out_csv)
print(sub.head())

# Download the CSV and the adapter weights
try:
    from google.colab import files
    files.download(str(out_csv)); files.download(str(OUT_DIR / "lora_best.pth"))
except Exception as e:
    print("Download manually from the file browser:", e)